Reprenons **depuis le début**, avec des mots très simples et des exemples de la vie courante.

Pour ne plus tout mélanger, imaginez une **grande grille Excel** qu'on donne à manger à nos modèles.

---

# 🧠 1. La métaphore du Tableau Excel (Le concept fondamental)

Dans votre projet, tout modèle d'IA travaille avec un tableau Excel :

```
             ┌──────────────────────────────────────────────────────────┐
             │            CARACTÉRISTIQUES (Les Colonnes)               │
             │   Séquence ADN   │   k-mer 1  │  k-mer 2  │ Taux GC ...  │
┌────────────┼──────────────────┼────────────┼───────────┼──────────────┤
│ ÉCHANTILLONS│ "ATGCGATCG..."   │    0.05    │   0.12    │    45.5%     │ ──► Ligne 1
│ (Les Lignes)│ "CGATCGATC..."   │    0.01    │   0.08    │    60.2%     │ ──► Ligne 2
│            │ "GCTAGCTAG..."   │    0.00    │   0.03    │    38.0%     │ ──► Ligne 3 ...
└────────────┴──────────────────┴────────────┴───────────┴──────────────┘
```

Distinguons maintenant **3 notions différentes** qui vous embrouillent :

| Notion | C'est quoi dans Excel ? | Nos chiffres dans le projet |
| :--- | :--- | :--- |
| **1. Les Échantillons** | Le nombre de **LIGNES** (nombre de séquences d'ADN) | **4 000** ou **58 552** |
| **2. Les Caractéristiques** (*Features*) | Le nombre de **COLONNES** (les informations décrivant chaque séquence) | **256** au départ, puis **5 459** |
| **3. Les Paramètres du Modèle** | Les **formules/poids internes** du cerveau de l'IA (ses "boutons" de réglage) | Ex: 8 257 ou 411 393 |

---

# 📄 2. D'où viennent 4 000 vs 58 552 ? (Les Lignes)

### D'où vient le `4 000` ?
Dans le notebook initial [03_knowledge_distillation.ipynb](file:///Users/user/PythonProjects/learning/EEIA-bioAI-Workshop-project/day3/03_knowledge_distillation.ipynb) (Cellule 1) :
```python
X_train_emb, y_train, ids_train = load_supervised_embeddings(EMB_DIR, "train")
```
L'organisateur du atelier a extrait les représentations vectorielles (*embeddings*) du gros modèle Evo2 pour **seulement 4 000 séquences** (enregistrées dans un fichier `train.npz`).
* **Pourquoi ?** Parce que calculer les représentations avec le modèle à 7 milliards de paramètres prend beaucoup de temps.
* C'est pourquoi par défaut, `load_split(...)` dans [data.py](file:///Users/user/PythonProjects/learning/EEIA-bioAI-Workshop-project/day3/src/data.py#L18) fixe `DEFAULT_MAX_ROWS = 4000`.

### D'où vient le `58 552` ?
Si vous ouvrez le fichier CSV brut qui contient toutes les données générées au Jour 1 (`2-data/processed/train.csv`), il y a en réalité **58 552 lignes** (58 552 séquences d'ADN) !
Dans le code de notre expérience (Cellule 2 du notebook ou script `run_round3.py`) :
```python
splits = load_all(PROCESSED_DIR, max_rows=None) # max_rows=None désactive la limite de 4000 !
```
En mettant `max_rows=None`, nous avons chargé le fichier entier : **58 552 séquences**. 
> **En résumé :** Au lieu de faire réviser le Student sur seulement 4 000 exercices (les 4 000 lignes notées par le Teacher), nous l'avons fait réviser sur **tous les 58 552 exercices disponibles** du livre.

---

# 🧬 3. D'où viennent les 256 vs 5 459 Caractéristiques ? (Les Colonnes)

Le modèle de machine learning ne sait pas lire du texte brut comme `"ATGCGATC..."`. Il lui faut des nombres dans les colonnes d'Excel.

### A. Au départ : 256 colonnes ($k=4$)
Dans [featurize.py](file:///Users/user/PythonProjects/learning/EEIA-bioAI-Workshop-project/day3/src/featurize.py#L28), la fonction `kmer_frequencies(seq, k=4)` découpe la séquence en sous-mots de 4 lettres (`AAAA`, `AAAC`, ..., `TTTT`).
* Il y a $4^4 = 4 \times 4 \times 4 \times 4 = \mathbf{256}$ combinaisons possibles de 4 bases.
* Chaque séquence d'ADN devenait donc une ligne avec **256 colonnes** (la fréquence de chaque mot de 4 lettres).

### B. À la fin : 5 459 colonnes (Notre amélioration du Round 3)
Dans notre script [run_round3.py](file:///Users/user/.gemini/antigravity-ide/brain/d5f0b820-7dcd-459b-9146-67471b0bed62/scratch/run_round3.py), nous avons fabriqué un tableau Excel beaucoup plus détaillé en additionnant plusieurs types de colonnes :

1. **Les 3-mers** ($4^3$) = **64 colonnes** (ex: `AAA`, `AAC`...)
2. **Les 4-mers** ($4^4$) = **256 colonnes** (ex: `AAAA`, `AAAC`...)
3. **Les 5-mers** ($4^5$) = **1 024 colonnes** (ex: `AAAAA`...)
4. **Les 6-mers** ($4^6$) = **4 096 colonnes** (ex: `AAAAAA`...)
5. **19 Caractéristiques Biologiques** calculées à la main :
   - `1` colonne pour le **Taux de GC** (pourcentage de bases G et C).
   - `16` colonnes pour les **Dinucléotides** (fréquence de `AA`, `AC`, `AG`, `AT`, `CA`...).
   - `1` colonne pour l'**Entropie** (la diversité/complexité chimique de la séquence).
   - `1` colonne pour la **Longueur du plus grand ORF** (cadre de lecture protéique).

**Faisons l'addition :**
$$64 + 256 + 1024 + 4096 + 19 = \mathbf{5\ 459\ \text{colonnes (caractéristiques)}}$$

> **En résumé :** Au lieu d'écrire seulement 256 détails sur chaque séquence, nous avons donné au modèle **5 459 indices différents** par séquence pour l'aider à deviner si c'est codant ou non.

---

# ⚙️ 4. C'est quoi alors les "Paramètres du Modèle" ?

Ne confondez pas les colonnes d'entrée ($X$) avec les **paramètres** ($W$).

### Une équation très simple pour comprendre :
Imaginons une formule de prédiction de prix de maison :
$$\text{Prix} = (\mathbf{w_1} \times \text{Surface}) + (\mathbf{w_2} \times \text{Nb\_Chambres}) + \mathbf{b}$$

* **Surface** et **Nb_Chambres** sont les **2 caractéristiques (colonnes d'entrée)**.
* $\mathbf{w_1}$, $\mathbf{w_2}$ et $\mathbf{b}$ sont les **3 paramètres (poids internes)** de l'équation. C'est ce que l'IA ajuste pendant l'entraînement !

Dans un réseau de neurones (comme `MLPHead` dans [classifier_heads.py](file:///Users/user/PythonProjects/learning/EEIA-bioAI-Workshop-project/day3/src/models/classifier_heads.py#L18)) :
* Si on lui donne **256 colonnes d'entrée** avec 32 neurones cachés, il y a **8 257 poids** à ajuster.
* Si on lui donne **5 459 colonnes d'entrée** avec 256 neurones cachés, il y a **411 393 poids** à ajuster.

---

# 🔗 5. Le lien direct avec le Notebook [03_knowledge_distillation.ipynb](file:///Users/user/PythonProjects/learning/EEIA-bioAI-Workshop-project/day3/03_knowledge_distillation.ipynb)

Reprenons le notebook que vous avez sous les yeux :

1. **Cellule Code 1 (Ligne 86-87)** :
   ```python
   X_train_emb, y_train, ids_train = load_supervised_embeddings(EMB_DIR, "train")

   
   ```
   * *Explication :* On charge le sous-ensemble de **4 000 lignes** notées par le Teacher.
2. **Cellule Code 2 (Ligne 142-143)** :
   ```python
   X_train_kmer = torch.tensor(kmer_matrix(train_seqs, k=4), dtype=torch.float32)
   ```
   * *Explication :* On calcule les 4-mers $\rightarrow$ **256 colonnes**.
3. **Cellule Code 4 (Ligne 217)** :
   ```python
   candidate_student = MLPHead(d_in=X_train_kmer.shape[1], d_hidden=d_hidden)
   ```
   * *Explication :* On crée un petit réseau avec `d_in = 256` colonnes. Score obtenu = **83.5%**.

### Ce qu'on a fait pour passer à 90.7% :
Nous avons écrit le script `run_round3.py` qui fait :
1. `max_rows=None` $\rightarrow$ Utiliser **58 552 lignes** au lieu de 4 000.
2. `multi_kmer_matrix(seqs, ks=(3,4,5,6)) + bio_features` $\rightarrow$ Créer **5 459 colonnes** au lieu de 256.
3. Entraîner un **Ensemble de 4 modèles** (RandomForest, GBM, LogReg, MLP) qui votent ensemble.
4. Score obtenu = **90.70%** !

Est-ce que la différence entre **lignes** (échantillons), **colonnes** (caractéristiques) et **poids** (paramètres) est plus claire pour vous maintenant ?